In [1]:
%load_ext autoreload
%autoreload 2
import torch
from dotenv import load_dotenv
from accelerate import Accelerator
from HuggingFaceModel import HuggingFaceModel
from TrainStrategy import TrainStrategy
from constant import *
from LlmOutputLabelConverter import LlmOutputLabelConverter

/Users/shahidul/dev/project/technical-debt/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/shahidul/dev/project/technical-debt/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
accelerator = Accelerator()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
prompt_template = PromptTemplate(
    name="Manually Crafted",
    definition="You are a Code Analysis Expert specialized in detecting Self-Admitted Technical Debt (SATD) in Java test code comments. SATD refers to comments where developers acknowledge that the current test implementation is incomplete, suboptimal, or relies on a compromise that should be addressed in the future. These admissions often appear as markers such as TODO or FIXME, or as notes about unresolved issues, temporary fixes, workarounds, hacks, performance limitations, use of deprecated APIs, unsupported features, poor design choices, skipped tests, or uncertain functionality. However, comments that only describe expected behavior of test code are not SATD unless there is additional information indicating the need for future improvement.",
    instruction="Think step by step and assign the label of yes or no for each given test code comment.",
    n_shot_template='Comment: {{ text }}',
    n_shot_answer_template="Answer: {{ cot }} The answer is {{ label }}.",
    line_m_before=3,
    line_n_after=3)
output_label_converter = LlmOutputLabelConverter({'yes', 'no'}, DEFAULT_DETECTION_CLASS)

In [ ]:
for model_name in ['google/flan-t5-small', 'google/flan-t5-base', 'google/flan-t5-large', 'google/flan-t5-xl']:
    for shots in [2*n for n in range(6)]:
        flan_t5_detection_model = HuggingFaceModel('detect', model_name, output_label_converter, False)
        flan_t5_detection_model.fit(detect_n_shot_dataset)
        flan_t5_detection_model.predict(detect_test_dataset, DATASET_NAME, prompt_template, TrainStrategy.N_SHOT_TOP, shots, verbose=False)


In [ ]:
for model_name in ['google/flan-t5-xxl']:
    for shots in [2*n for n in range(6)]:
        flan_t5_detection_model = HuggingFaceModel('detect', model_name, output_label_converter,True)
        flan_t5_detection_model.fit(detect_n_shot_dataset)
        flan_t5_detection_model.predict(detect_test_dataset.select(range(10)), DATASET_NAME, prompt_template, TrainStrategy.N_SHOT_TOP, shots, verbose=False)


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]